# Forest Fire AI — Tabular Model Evaluation
## Notebook 09: Full Evaluation with Metrics, Confusion Matrix, SHAP


In [ ]:
import os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, mean_absolute_error,
                              mean_squared_error, r2_score, confusion_matrix,
                              classification_report, roc_curve)

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
META_DIR = IMPL / "artifacts" / "metadata"
PLOTS    = IMPL / "artifacts" / "plots"
METRICS  = IMPL / "artifacts" / "metrics"
DATA_DIR = IMPL / "data" / "processed"
TAB_DIR  = IMPL / "models" / "tabular"

clf    = pickle.load(open(TAB_DIR / "classifier.pkl",  "rb"))
reg    = pickle.load(open(TAB_DIR / "regressor.pkl",   "rb"))
scaler = pickle.load(open(TAB_DIR / "scaler.pkl",      "rb"))
le_month = pickle.load(open(TAB_DIR / "le_month.pkl",  "rb"))
le_day   = pickle.load(open(TAB_DIR / "le_day.pkl",    "rb"))
with open(TAB_DIR / "metadata.json") as f:
    tmeta = json.load(f)

FEATURE_COLS = tmeta['feature_cols']
NUM_FEATS    = tmeta['numerical_features']

df = pd.read_csv(DATA_DIR / "forestfires_processed.csv")
df['month_enc'] = le_month.transform(df['month'])
df['day_enc']   = le_day.transform(df['day'])

X = df[FEATURE_COLS].values
y_cls = df['fire_occurred'].values
y_reg = np.log1p(df['area'].values)

SEED = 42
_, X_te, _, yc_te, _, yr_te = train_test_split(
    X, y_cls, y_reg, test_size=0.20, random_state=SEED, stratify=y_cls
)

print(f"Test samples: {len(X_te)}")
print(f"Classifier:  {tmeta['classifier_name']}")
print(f"Regressor:   {tmeta['regressor_name']}")


In [ ]:
# Classification evaluation
yc_pred = clf.predict(X_te)
yc_prob = clf.predict_proba(X_te)[:, 1]

acc  = accuracy_score(yc_te, yc_pred)
prec = precision_score(yc_te, yc_pred, zero_division=0)
rec  = recall_score(yc_te, yc_pred, zero_division=0)
f1   = f1_score(yc_te, yc_pred, zero_division=0)
auc  = roc_auc_score(yc_te, yc_prob)
cm   = confusion_matrix(yc_te, yc_pred)

print("="*50)
print("CLASSIFICATION EVALUATION (fire_occurred)")
print("="*50)
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  ROC-AUC:   {auc:.4f}")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(yc_te, yc_pred, target_names=['No Fire','Fire']))


In [ ]:
# Regression evaluation
yr_pred_log = reg.predict(X_te)
yr_pred     = np.expm1(yr_pred_log)
yr_te_orig  = np.expm1(yr_te)

mae  = mean_absolute_error(yr_te_orig, yr_pred)
mse  = mean_squared_error(yr_te_orig, yr_pred)
rmse = np.sqrt(mse)
r2   = r2_score(yr_te, yr_pred_log)

print("REGRESSION EVALUATION (area in ha, log-space R²)")
print(f"  MAE:  {mae:.4f} ha")
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f} ha")
print(f"  R²:   {r2:.4f}")

tab_test_metrics = {
    'classifier': {'accuracy':round(acc,4),'precision':round(prec,4),
                   'recall':round(rec,4),'f1':round(f1,4),'roc_auc':round(auc,4)},
    'regressor':  {'mae':round(mae,4),'mse':round(mse,4),'rmse':round(rmse,4),'r2':round(r2,4)}
}
with open(METRICS / "tabular_test_metrics.json", "w") as f:
    json.dump(tab_test_metrics, f, indent=2)


In [ ]:
# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\n{tmeta["classifier_name"]}', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['No Fire','Fire']); axes[0].set_yticklabels(['No Fire','Fire'])
plt.colorbar(im, ax=axes[0])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=14, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(yc_te, yc_prob)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.5)
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Predicted vs Actual (log scale)
axes[2].scatter(yr_te, yr_pred_log, alpha=0.5, color='#2E8B57', edgecolors='k', linewidths=0.5)
mn, mx = min(yr_te.min(), yr_pred_log.min()), max(yr_te.max(), yr_pred_log.max())
axes[2].plot([mn, mx], [mn, mx], 'r--', lw=2)
axes[2].set_title(f'Actual vs Predicted (log area)\nR²={r2:.4f}', fontweight='bold')
axes[2].set_xlabel('Actual log(area+1)'); axes[2].set_ylabel('Predicted log(area+1)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS / "tabular_evaluation.png", dpi=100, bbox_inches='tight')
plt.close()
print("Tabular evaluation plots saved.")


In [ ]:
# SHAP values (if available)
try:
    import shap
    explainer = shap.TreeExplainer(clf)
    shap_vals = explainer.shap_values(X_te)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    shap_importance = np.abs(shap_vals).mean(axis=0)
    si_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': shap_importance})
    si_df = si_df.sort_values('importance', ascending=True)
    ax.barh(si_df['feature'], si_df['importance'], color='#DC143C', edgecolor='black', alpha=0.8)
    ax.set_title(f'SHAP Feature Importance — {tmeta["classifier_name"]}', fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(PLOTS / "shap_importance.png", dpi=100, bbox_inches='tight')
    plt.close()
    print("SHAP importance plot saved.")
except Exception as e:
    print(f"SHAP skipped (optional): {e}")

print("\nNotebook 09 complete.")
